# 06 | Treinar e comparar as previsões

Usamos **aprendizado supervisionado**: mostramos ao programa exemplos com a resposta conhecida para que aprenda a prever a taxa do próximo ano. Vamos comparar uma regra simples, o Modelo A e o Modelo B.

**Como usar:** execute os blocos de código 1, 2 e 3, nessa ordem. Leia a explicação antes de cada bloco. Cada bloco reúne pequenas ações que fazem parte da mesma etapa.

## Bloco 1: Abrir e escolher as entradas

O Modelo A usa homicídios anteriores e economia. O B usa isso e mais quatro colunas de leis. A resposta é a taxa de homicídios do próximo ano.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
base = pd.read_csv(ROOT/"data/processed/us_gun_policy_supervised_v3.csv")
a = ["homicide_rate","unemployment_rate","income_per_capita"]
b = a+["ccw_permitless","ccw_may_issue","background_handgun","background_longgun"]
alvo = "homicide_rate_next_year"
print("Base:",base.shape,"| Modelo A:",len(a),"entradas | B:",len(b),"entradas")

## Bloco 2: Treinar e testar em três rodadas

**Janela expansiva**: primeiro usamos poucos anos para treinar; em cada rodada, acrescentamos mais um ano. Sempre avaliamos em um ano posterior. O MAE mede o erro médio; o RMSE dá peso maior aos erros grandes.

In [ ]:
resultados = []
for ano_teste in (2021,2022,2023):
    treino = base[base["year"] < ano_teste]
    teste = base[base["year"] == ano_teste]
    y = teste[alvo]
    for nome, colunas in [("Modelo A",a),("Modelo B",b)]:
        modelo = Pipeline([("scaler",StandardScaler()),("regressao",LinearRegression())])
        modelo.fit(treino[colunas],treino[alvo])
        previsao = modelo.predict(teste[colunas])
        resultados.append({"ano_previsto":ano_teste+1,"modelo":nome,
            "MAE":mean_absolute_error(y,previsao),
            "RMSE":np.sqrt(mean_squared_error(y,previsao))})
    simples = teste["homicide_rate"]
    resultados.append({"ano_previsto":ano_teste+1,"modelo":"Ingênuo",
        "MAE":mean_absolute_error(y,simples),
        "RMSE":np.sqrt(mean_squared_error(y,simples))})
detalhes = pd.DataFrame(resultados).sort_values(["ano_previsto","modelo"])
resumo = detalhes.groupby("modelo")[["MAE","RMSE"]].mean().sort_values("MAE")
display(detalhes.round(3));display(resumo.round(3))

## Bloco 3: Salvar e conferir

Os resultados devem ficar **próximos** aos do relatório quando usarmos as mesmas versões das fontes. Se houver diferença, verificamos os dados antes de concluir que o código está errado.

In [ ]:
saida = ROOT / "reports/results"
saida.mkdir(parents=True,exist_ok=True)
detalhes.to_csv(saida/"temporal_validation.csv",index=False)
resumo.round(3).to_csv(saida/"model_comparison_summary.csv")
print("Arquivos de resultados atualizados.")
print("Referência histórica (MAE médio): Ingênuo 0,785 | A 0,944 | B 0,959")
print("Veja também o RMSE médio esperado: 1,027 | 1,239 | 1,263")